<a href="https://colab.research.google.com/github/Abhilipsha-behera22/Multi-Agent-System-With-LangGraph/blob/main/project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install and Authenticate**

In [39]:
# Install the necessary libraries
!pip install -qU langgraph langchain-google-genai langchain-community duckduckgo-search ddgs

In [18]:
import os
from google.colab import userdata

# This pulls the key from your Secrets tab automatically
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

print("Environment ready!")

Environment ready!


**Define the Agents and Workflow**

In [49]:
from typing import TypedDict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

In [50]:
# --- Shared State ---
class AgentState(TypedDict):
    topic: str
    research_data: List[str]
    blog_post: str

In [51]:
# --- Researcher ---
def researcher_node(state: AgentState):
    print(f"Researcher: Searching for facts about '{state['topic']}'...")
    search = DuckDuckGoSearchRun()
    try:
        results = search.run(f"latest 2026 facts about {state['topic']}")
    except Exception as e:
        results = f"Search failed: {e}"
    return {"research_data": [results]}

In [52]:
# --- Writer (Updated for April 2026) ---
def writer_node(state: AgentState):
    print("Writer: Generating blog post...")

    # gemini-2.5-flash is the current stable choice for speed/accuracy
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

    data = state["research_data"][-1] if state["research_data"] else "No data."

    prompt = ChatPromptTemplate.from_template(
        "Write a 3-paragraph blog post about {topic} using: {data}. Use Markdown."
    )

    chain = prompt | llm
    response = chain.invoke({"topic": state["topic"], "data": data})

    # Helper to handle the new 2026 'content block' format if necessary
    content = response.content
    if isinstance(content, list):
        content = content[0].get('text', str(content))

    return {"blog_post": content}

In [53]:
# --- Graph Assembly ---
builder = StateGraph(AgentState)
builder.add_node("Researcher", researcher_node)
builder.add_node("Writer", writer_node)
builder.set_entry_point("Researcher")
builder.add_edge("Researcher", "Writer")
builder.add_edge("Writer", END)
app = builder.compile()
print("Workflow compiled with Gemini 2.5 Flash!")

Workflow compiled with Gemini 2.5 Flash!


**Run the Project**

In [54]:
from IPython.display import Markdown

inputs = {
    "topic": "The rise of AI Wearables in 2026",
    "research_data": [],
    "blog_post": ""
}

print("Starting the Multi-Agent System...")
result = app.invoke(inputs)

print("\n" + "="*30 + "\n")
Markdown(result["blog_post"])

Starting the Multi-Agent System...
Researcher: Searching for facts about 'The rise of AI Wearables in 2026'...
Writer: Generating blog post...




## The AI Wearable Revolution: A Look at 2026's Smartest Devices

As of early 2026, the landscape of AI wearables has undergone a profound transformation, moving beyond mere fitness trackers to become integral components of our daily lives. A comprehensive look at the industry on February 12, 2026, revealed an exciting array of devices, from sophisticated smart glasses to advanced health monitors, all leveraging artificial intelligence to enhance user experience. This expansion was clearly evident at MWC 2026 in early March, where wearables continued to solidify their role within the broader mobile ecosystem. While smartwatches showed signs of maturity and slower growth, AI-powered smart glasses and innovative clip-on TWS earbuds demonstrated strong momentum, signaling a significant shift in consumer interest and technological focus.

The defining characteristic of AI wearable devices in 2026 is an architectural shift towards local Edge AI. This critical development, highlighted in "The ultimate guide to AI wearable devices 2026" on March 25, has drastically reduced latency to milliseconds and, crucially, eliminated mandatory subscription models for core functionality. A detailed guide published on March 30 further elaborates on every major AI wearable category, explaining what each device type truly does, which models offer genuine value, and precisely what AI genuinely adds to the user experience. This technological leap allows for more responsive, private, and independent devices, truly unlocking the potential of AI without constant cloud dependency.

From the exciting announcements at MWC 2026, including advancements in Android XR and Wear OS watches, it's clear that the industry is rapidly evolving. Smart Analytics Global (SAG) observed the strong momentum in AI smart glasses and clip-on TWS earbuds, marking them as the leading edge of innovation. While some categories may still be overpromising, the core advancements in Edge AI and the genuine utility offered by devices across health, communication, and augmented reality signal a bright future. The best wearables and announcements from MWC 2026 underscore a future where AI isn't just a feature, but the foundational intelligence that makes our wearable technology truly smart and indispensable.